# Wireless Security Auditor

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
#!/usr/bin/env python3

import subprocess
import sys
import os
import re
import json
import time
import threading
from datetime import datetime
from typing import Dict, List, Optional, Tuple
import warnings
warnings.filterwarnings('ignore')

class WirelessSecurityAuditor:
    def __init__(self):
        self.results = {}
        self.scan_data = []
        self.vulnerabilities = []
        self.interface = None
        self.is_monitoring = False

    def install_dependencies(self):
        packages = [
            'scapy',
            'python-nmap',
            'requests',
            'beautifulsoup4',
            'netifaces'
        ]

        for package in packages:
            try:
                subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
                print(f"{package} installed successfully")
            except subprocess.CalledProcessError:
                print(f"Failed to install {package}")

        try:
            os.system('apt-get update -qq')
            os.system('apt-get install -y aircrack-ng wireless-tools net-tools -qq')
            print("System tools installed")
        except:
            print("System tools installation may require elevated privileges")

    def check_environment(self):
        try:
            import google.colab
            print("Google Colab environment detected")
            return True
        except ImportError:
            print("Not running in Google Colab")
            return False

    def simulate_network_scan(self):
        print("Simulating wireless network scan...")

        simulated_networks = [
            {
                'ssid': 'HomeNetwork_5G',
                'bssid': '00:11:22:33:44:55',
                'channel': 11,
                'signal': -45,
                'encryption': 'WPA2',
                'security_score': 7
            },
            {
                'ssid': 'OpenWiFi',
                'bssid': '00:11:22:33:44:66',
                'channel': 6,
                'signal': -55,
                'encryption': 'Open',
                'security_score': 1
            },
            {
                'ssid': 'OldRouter',
                'bssid': '00:11:22:33:44:77',
                'channel': 1,
                'signal': -65,
                'encryption': 'WEP',
                'security_score': 2
            },
            {
                'ssid': 'Enterprise_WiFi',
                'bssid': '00:11:22:33:44:88',
                'channel': 36,
                'signal': -40,
                'encryption': 'WPA2-Enterprise',
                'security_score': 9
            }
        ]

        self.scan_data = simulated_networks
        print(f"Found {len(simulated_networks)} networks")
        return simulated_networks

    def analyze_security(self, network: Dict) -> Dict:
        vulnerabilities = []
        recommendations = []

        if network['encryption'] == 'Open':
            vulnerabilities.append({
                'type': 'No Encryption',
                'severity': 'Critical',
                'description': 'Network is completely open and unencrypted'
            })
            recommendations.append('Enable WPA3 or WPA2 encryption')

        elif network['encryption'] == 'WEP':
            vulnerabilities.append({
                'type': 'Weak Encryption',
                'severity': 'High',
                'description': 'WEP encryption is easily breakable'
            })
            recommendations.append('Upgrade to WPA3 or WPA2')

        elif network['encryption'] == 'WPA':
            vulnerabilities.append({
                'type': 'Outdated Encryption',
                'severity': 'Medium',
                'description': 'WPA is vulnerable to dictionary attacks'
            })
            recommendations.append('Upgrade to WPA3 or WPA2')

        if network['signal'] > -30:
            vulnerabilities.append({
                'type': 'Strong Signal',
                'severity': 'Low',
                'description': 'Network has very strong signal, potentially extending range'
            })
            recommendations.append('Reduce transmission power if not needed')

        if network['channel'] in [1, 6, 11]:
            vulnerabilities.append({
                'type': 'Common Channel',
                'severity': 'Low',
                'description': 'Using common 2.4GHz channel prone to interference'
            })
            recommendations.append('Consider using 5GHz band or less crowded channels')

        return {
            'vulnerabilities': vulnerabilities,
            'recommendations': recommendations,
            'risk_level': self.calculate_risk_level(vulnerabilities)
        }

    def calculate_risk_level(self, vulnerabilities: List[Dict]) -> str:
        if any(v['severity'] == 'Critical' for v in vulnerabilities):
            return 'Critical'
        elif any(v['severity'] == 'High' for v in vulnerabilities):
            return 'High'
        elif any(v['severity'] == 'Medium' for v in vulnerabilities):
            return 'Medium'
        else:
            return 'Low'

    def perform_nmap_scan(self, target_range: str = "192.168.1.0/24"):
        try:
            import nmap
            nm = nmap.PortScanner()
            print(f"Scanning network range: {target_range}")
            result = nm.scan(target_range, arguments='-sn')

            hosts = []
            for host in nm.all_hosts():
                if nm[host].state() == 'up':
                    hosts.append({
                        'ip': host,
                        'hostname': nm[host].hostname(),
                        'state': nm[host].state()
                    })

            return hosts
        except ImportError:
            print("python-nmap not available, skipping network scan")
            return []
        except Exception as e:
            print(f"Network scan failed: {e}")
            return []

    def detect_rogue_access_points(self, networks: List[Dict]) -> List[Dict]:
        rogue_aps = []

        known_networks = ['HomeNetwork_5G', 'Enterprise_WiFi']

        for network in networks:
            if network['ssid'] not in known_networks:
                if network['encryption'] == 'Open':
                    rogue_aps.append({
                        'ssid': network['ssid'],
                        'bssid': network['bssid'],
                        'reason': 'Unknown open network',
                        'risk': 'High'
                    })
                elif network['ssid'] in ['Free WiFi', 'Public WiFi', 'Guest']:
                    rogue_aps.append({
                        'ssid': network['ssid'],
                        'bssid': network['bssid'],
                        'reason': 'Suspicious public network name',
                        'risk': 'Medium'
                    })

        return rogue_aps

    def analyze_channel_utilization(self, networks: List[Dict]) -> Dict:
        channel_usage = {}

        for network in networks:
            channel = network['channel']
            if channel not in channel_usage:
                channel_usage[channel] = []
            channel_usage[channel].append(network['ssid'])

        congested_channels = {
            ch: networks for ch, networks in channel_usage.items()
            if len(networks) > 2
        }

        return {
            'channel_usage': channel_usage,
            'congested_channels': congested_channels,
            'recommendations': self.get_channel_recommendations(channel_usage)
        }

    def get_channel_recommendations(self, channel_usage: Dict) -> List[str]:
        recommendations = []

        if 6 in channel_usage and len(channel_usage[6]) > 2:
            recommendations.append("Channel 6 is congested, consider using channel 1 or 11")

        if any(len(networks) > 3 for networks in channel_usage.values()):
            recommendations.append("Multiple channels are congested, consider using 5GHz band")

        return recommendations

    def check_weak_passwords(self, networks: List[Dict]) -> List[Dict]:
        weak_password_indicators = []

        for network in networks:
            if network['encryption'] in ['WPA', 'WPA2']:
                weak_indicators = []

                if network['ssid'].lower() in ['admin', 'password', 'default']:
                    weak_indicators.append('Default/weak SSID')

                if len(network['ssid']) < 8:
                    weak_indicators.append('Short SSID may indicate weak password')

                if weak_indicators:
                    weak_password_indicators.append({
                        'ssid': network['ssid'],
                        'bssid': network['bssid'],
                        'indicators': weak_indicators
                    })

        return weak_password_indicators

    def generate_report(self) -> Dict:
        if not self.scan_data:
            return {"error": "No scan data available"}

        report = {
            'timestamp': datetime.now().isoformat(),
            'total_networks': len(self.scan_data),
            'network_analysis': [],
            'rogue_aps': self.detect_rogue_access_points(self.scan_data),
            'channel_analysis': self.analyze_channel_utilization(self.scan_data),
            'weak_passwords': self.check_weak_passwords(self.scan_data),
            'summary': {}
        }

        risk_levels = []
        for network in self.scan_data:
            analysis = self.analyze_security(network)
            network_report = {
                'ssid': network['ssid'],
                'bssid': network['bssid'],
                'encryption': network['encryption'],
                'signal': network['signal'],
                'channel': network['channel'],
                'analysis': analysis
            }
            report['network_analysis'].append(network_report)
            risk_levels.append(analysis['risk_level'])

        report['summary'] = {
            'critical_risks': risk_levels.count('Critical'),
            'high_risks': risk_levels.count('High'),
            'medium_risks': risk_levels.count('Medium'),
            'low_risks': risk_levels.count('Low'),
            'rogue_aps_detected': len(report['rogue_aps']),
            'congested_channels': len(report['channel_analysis']['congested_channels'])
        }

        return report

    def print_report(self, report: Dict):
        print("\n" + "="*60)
        print("WIRELESS SECURITY AUDIT REPORT")
        print("="*60)
        print(f"Timestamp: {report['timestamp']}")
        print(f"Total Networks Scanned: {report['total_networks']}")

        print("\nSUMMARY:")
        print(f"Critical Risks: {report['summary']['critical_risks']}")
        print(f"High Risks: {report['summary']['high_risks']}")
        print(f"Medium Risks: {report['summary']['medium_risks']}")
        print(f"Low Risks: {report['summary']['low_risks']}")
        print(f"Rogue APs Detected: {report['summary']['rogue_aps_detected']}")
        print(f"Congested Channels: {report['summary']['congested_channels']}")

        print("\nNETWORK ANALYSIS:")
        for network in report['network_analysis']:
            print(f"\nSSID: {network['ssid']}")
            print(f"BSSID: {network['bssid']}")
            print(f"Encryption: {network['encryption']}")
            print(f"Signal: {network['signal']} dBm")
            print(f"Channel: {network['channel']}")
            print(f"Risk Level: {network['analysis']['risk_level']}")

            if network['analysis']['vulnerabilities']:
                print("Vulnerabilities:")
                for vuln in network['analysis']['vulnerabilities']:
                    print(f"  - {vuln['type']} ({vuln['severity']}): {vuln['description']}")

            if network['analysis']['recommendations']:
                print("Recommendations:")
                for rec in network['analysis']['recommendations']:
                    print(f"  - {rec}")

        if report['rogue_aps']:
            print("\nROGUE ACCESS POINTS:")
            for rogue in report['rogue_aps']:
                print(f"SSID: {rogue['ssid']} | BSSID: {rogue['bssid']}")
                print(f"Reason: {rogue['reason']} | Risk: {rogue['risk']}")

        if report['channel_analysis']['congested_channels']:
            print("\nCHANNEL CONGESTION:")
            for channel, networks in report['channel_analysis']['congested_channels'].items():
                print(f"Channel {channel}: {len(networks)} networks")
                print(f"Networks: {', '.join(networks)}")

        print("\n" + "="*60)

    def run_audit(self):
        print("Starting Wireless Security Audit")

        self.check_environment()
        self.install_dependencies()

        networks = self.simulate_network_scan()

        print("\nPerforming network discovery...")
        hosts = self.perform_nmap_scan()
        if hosts:
            print(f"Found {len(hosts)} active hosts")

        print("\nGenerating security report...")
        report = self.generate_report()

        self.print_report(report)

        return report

def main():
    auditor = WirelessSecurityAuditor()
    report = auditor.run_audit()

    print("\nAudit complete. Report generated successfully.")
    return report

if __name__ == "__main__":
    main()